# Tardis Project

We, Loup, Lukas, and Eva, are part of a newly formed **SNCF Data Analysis Service**, dedicated to improving the efficiency of train travel across the country.

Our mission? Analyze historical train delay data, uncover hidden patterns, and develop a predictive model that can forecast delays before they happen. The SNCF has entrusted our team with making the railway system more efficient and transparent.

If we <span style="color:green">succeed</span>, our dashboard will be used by thousands of travelers to better plan their journeys. If we <span style="color:red">fail</span>... well, we expect a lot more unhappy commuters. 

No pressure! Using the provided dataset, our job is to clean and analyze historical delay data, develop a simple predictive model, and present our insights through an interactive **Streamlit dashboard**.

In [ ]:
# Importing the libraries to manage the dataset and visualize data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# --- CLEANING UTILITY FUNCTIONS ---

def clean_numeric_column(df, column_name, text_to_remove=None, to_type='float'):
    """
    Cleans a column surgically without using 'coerce'.
    Extracts the first valid numeric sequence found in the cell.
    """
    if column_name not in df.columns:
        return df
    
    # 1. Prepare work buffer as string
    processed_col = df[column_name].astype(str)
    
    # 2. Remove specific text if provided ("min")
    if text_to_remove is not None:
        processed_col = processed_col.str.replace(text_to_remove, "", regex=False)
    
    # 3. Normalization (commas -> dots, strip spaces)
    processed_col = processed_col.str.replace(",", ".", regex=False).str.strip()
    
    # 4. SECURITY FILTER: Extract the first valid number
    # Pattern: [optional sign] [digits] [optional dot] [digits]
    processed_col = processed_col.str.extract(r'([-+]?\d*\.?\d+)')[0]
    
    # 5. Final conversion (errors='raise' to guarantee cleaning quality and catch errors)
    df[column_name] = pd.to_numeric(processed_col, errors='raise')
    
    # 6. Final typing
    if to_type == 'int':
        df[column_name] = df[column_name].round().astype('Int64')
    else:
        df[column_name] = df[column_name].astype(float)
        
    return df

def drop_invalid_rows(df, column_name):
    """ Removes rows where the value is null or unusable. """
    return df.dropna(subset=[column_name])

def impute_missing_with_value(df, column_name, value):
    """ Replaces missing values with a fixed value (e.g., 0). """
    df[column_name] = df[column_name].fillna(value)
    return df

def impute_missing_with_mean(df, column_name):
    """ Replaces missing values with the column mean. """
    mean_val = df[column_name].mean()
    df[column_name] = df[column_name].fillna(mean_val)
    return df

# Step 1: Data Exploration and Cleaning
## 1. Loading the Dataset

Examining column names, count, and data types.

In [ ]:
df = pd.read_csv("dataset.csv", sep=";")
print("Column names and types:")

# Displaying column types
for col in df.columns:
    print(f"{col} -> {df[col].dtype}")

### Initial Data Inspection
We examine raw dataset statistics and preview the data.

In [ ]:
display(df.describe())

print("\nInitial dataset preview:")
display(df.head())

nb_line_start, nb_col_start = df.shape
print(f"\nThe original dataset contains {nb_col_start} columns and {nb_line_start} rows.")

### Visualizing Missing Values
Before cleaning, we identify missing values in the raw data using a heatmap.

In [ ]:
plt.figure(figsize=(15, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap (Raw Data)')
plt.show()

## 2. Duplicate Removal
We remove identical rows to ensure data integrity.

In [ ]:
df = df.drop_duplicates()

nb_line_after_dupes, _ = df.shape
print(f"Rows after duplicate removal: {nb_line_after_dupes} ({nb_line_start - nb_line_after_dupes} rows removed).")

## 3. Data Cleaning and Column Formatting

### Date Column
We standardize the `Date` column and extract `Year` and `Month` features.

In [ ]:
# Convert to datetime and standardize format
df['Date'] = pd.to_datetime(df['Date'], yearfirst=True, format='mixed')

# Drop rows with missing dates (critical for analysis)
df = drop_invalid_rows(df, 'Date')

# Extract Year and Month, then drop the original Date column
df['Year'] = df['Date'].dt.year.astype(int)
df['Month'] = df['Date'].dt.month.astype(int)
df = df.drop(columns=['Date'])

print(f"Remaining rows after date validation: {len(df)}")
display(df[['Year', 'Month']].head())

### Season Feature
We create a `Season` column based on the month.

In [ ]:
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    if month in [3, 4, 5]: return 'Spring'
    if month in [6, 7, 8]: return 'Summer'
    return 'Autumn'

df['Season'] = df['Month'].apply(get_season)
display(df[['Month', 'Season']].head())

### Service Column
We ensure the `Service` column only contains 'National' or 'International' values. Other values are set to `NaN`.

In [ ]:
# List of valid service categories for the SNCF network
valid_services = ['national', 'international']

# 1. Convert to lowercase and strip whitespace for uniform comparison
df['Service'] = df['Service'].str.lower().str.strip()

# 2. Apply a transformation to each cell using a 'lambda' function:
# - If the value is valid, capitalize it (e.g., 'national' -> 'National')
# - If the value is invalid/unknown, set it to 'pd.NA' (Missing Value)
df['Service'] = df['Service'].apply(lambda x: x.capitalize() if x in valid_services else pd.NA)

# Visualize the distribution of service types to understand dataset balance
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Service', hue='Service', palette='viridis', legend=False)
plt.title('Distribution of Train Services')
plt.ylabel('Count')
plt.show()

print(f"Rows with missing or invalid Service values: {df['Service'].isna().sum()}")
print("Service counts:")

### Departure and Arrival Stations
We convert station names to uppercase and remove rows with missing station information.

In [ ]:
for col in ['Departure station', 'Arrival station']:
    # Standardize to uppercase and strip spaces
    df[col] = df[col].str.upper().str.strip()
    
    # Replace null values with Pandas NA
    df[col] = df[col].replace(['NAN', 'NULL'], pd.NA)
    
    # Remove rows where station name is missing
    df = drop_invalid_rows(df, col)

# Visualize Top 10 most frequent Departure Stations
plt.figure(figsize=(12, 6))
top_stations = df['Departure station'].value_counts().nlargest(10)
sns.barplot(x=top_stations.index, y=top_stations.values, hue=top_stations.index, palette='rocket', legend=False)
plt.title('Top 10 Departure Stations by Traffic Volume')
plt.xticks(rotation=45)
plt.ylabel('Number of Scheduled Trains')
plt.show()

print(f"Final count after station validation: {len(df)}")
print("\nUnique values summary:")
print(df[['Service', 'Departure station', 'Arrival station', 'Season']].nunique())

### Average Journey Time
We clean the journey time by removing the 'min' text and converting the values to integers. Missing values are kept as `NaN` for now, pending a strategic decision on imputation.

In [ ]:
# Clean numeric data: remove 'min', handle commas, and convert to Integer
df = clean_numeric_column(df, 'Average journey time', text_to_remove='min', to_type='int')

# TODO: Revisit null values for 'Average journey time'. Currently left as NaN.

# Visualize distribution using a Violin Plot for a more modern aesthetic
plt.figure(figsize=(10, 6))
sns.violinplot(x=df['Average journey time'], color='lightgreen', inner='quartile')
plt.title('Distribution of Average Journey Time (Violin Plot)')
plt.xlabel('Journey Time (minutes)')
plt.ylabel('Number of trains')
plt.show()

print(f"Missing values in Average journey time: {df['Average journey time'].isna().sum()}")

### Scheduled and Cancelled Trains
We standardize the number of trains. We also implement a validation rule: the number of cancelled trains cannot exceed the number of scheduled trains.

In [ ]:
# Clean Scheduled and Cancelled columns
df = clean_numeric_column(df, 'Number of scheduled trains', to_type='int')
df = clean_numeric_column(df, 'Number of cancelled trains', to_type='int')

# Validation: Cancelled trains cannot exceed Scheduled trains
invalid_mask = df['Number of cancelled trains'] > df['Number of scheduled trains']
nb_invalid = invalid_mask.sum()

if nb_invalid > 0:
    print(f"WARNING: {nb_invalid} rows found where cancellations exceed scheduled trains. Setting these cancellation counts to NULL.")
    # TODO: Need to determine what to do with invalid values
    df.loc[invalid_mask, 'Number of cancelled trains'] = pd.NA

print(f"Missing 'Scheduled': {df['Number of scheduled trains'].isna().sum()}")
print(f"Missing 'Cancelled': {df['Number of cancelled trains'].isna().sum()}")

### Number of trains delayed at departure

Correct the format of the number of trains delayed at departure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed at departure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Convert column to int because it can't be other for a count

In [ ]:
df["Number of trains delayed at departure"] = pd.to_numeric(df["Number of trains delayed at departure"].str.replace(",", "."), downcast='integer')

### Average delay of late trains at departure

Correct the format of the average delay of late trains at departure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of late trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of late trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float, because we have here averages

In [ ]:
df["Average delay of late trains at departure"] = pd.to_numeric(df["Average delay of late trains at departure"].str.replace(",", "."), downcast='float')
df["Average delay of late trains at departure"] = df["Average delay of late trains at departure"].round(3)

### Average delay of all trains at departure

Correct the format of the number of the average delay of all trains at departure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of all trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of all trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float because we have an average, and to be precise it's better to have floats

In [ ]:
df["Average delay of all trains at departure"] = pd.to_numeric(df["Average delay of all trains at departure"].str.replace(",", "."), downcast='float')
df["Average delay of all trains at departure"] = df["Average delay of all trains at departure"].round(3)

### Departure delay comments

Departure delay comments removed because there is no data inside it

In [ ]:
df = df.drop("Departure delay comments", axis=1)

print("New column names are: ")
#We get names of columns in the Dataframe
columns = df.columns
for i in range(len(columns) - 1):
    print(columns[i], end=", ")
print(columns[-1])

### Number of trains delayed at arrival

Correct the format of the number of the number of trains delayed at arrival

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed at arrival"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Convert to int because it's a count

In [ ]:
df["Number of trains delayed at arrival"] = pd.to_numeric(df["Number of trains delayed at arrival"].str.replace(",", "."), downcast='integer')

### Average delay of late trains at arrival

Correct the format of the average delay of late trains at arrival

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of late trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of late trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float, because we have an average

In [ ]:
df["Average delay of late trains at arrival"] = pd.to_numeric(df["Average delay of late trains at arrival"].str.replace(",", "."), downcast='integer')

### Average delay of all trains at arrival

Correct the format of the average delay of all trains at arrival

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)


In [ ]:
values = df["Average delay of all trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of all trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float because it's an average

In [ ]:
df["Average delay of all trains at arrival"] = pd.to_numeric(df["Average delay of all trains at arrival"].str.replace(",", "."), downcast='integer')

### Arrival delay comments

Change every empty cells or cells without comments (NC, NA Non communiqué...)

In [ ]:
values = df["Arrival delay comments"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]) or str(values.iloc[i]).lower().strip() in ["nc", "na", "non communiqué", "null", "nan"]:
        values.iloc[i] = "NC"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Number of trains delayed > 15min

Correct the format of the number of trains delayed > 15min

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed > 15min"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Number of trains delayed > 15min"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

3. Convert column to int because it's a quantity

In [ ]:
df["Number of trains delayed > 15min"] = pd.to_numeric(df["Number of trains delayed > 15min"].str.replace(",", "."), downcast='integer')

### Average delay of trains > 15min (if competing with flights)

Correct the format of the average delay of trains > 15min (if competing with flights)

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of trains > 15min (if competing with flights)"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove "min" mention

In [ ]:
values = df["Average delay of trains > 15min (if competing with flights)"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

print(f"{nb_modified} cells has been modified.")

3. Convert column to float because it's averages

In [ ]:
df["Average delay of trains > 15min (if competing with flights)"] = pd.to_numeric(df["Average delay of trains > 15min (if competing with flights)"].str.replace(",", "."), downcast='float')
df["Average delay of trains > 15min (if competing with flights)"] = df["Average delay of trains > 15min (if competing with flights)"].round(3)

### Number of trains delayed > 30min

Correct the format of the number of trains delayed > 30min

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed > 30min"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Number of trains delayed > 30min"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

3. Convert column to int because it's a quantity

In [ ]:
df["Number of trains delayed > 30min"] = pd.to_numeric(df["Number of trains delayed > 30min"].str.replace(",", "."), downcast='integer')

### Number of trains delayed > 60min

Correct the format of the number of trains delayed > 60min

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed > 60min"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Number of trains delayed > 60min"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

3. Convert column to int because it's a quantity

In [ ]:
df["Number of trains delayed > 60min"] = pd.to_numeric(df["Number of trains delayed > 60min"].str.replace(",", "."), downcast='integer')

### Pct delay due to external causes

Correct the format of the Pct delay due to external causes

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to external causes"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove every "%"

In [ ]:
values = df["Pct delay due to external causes"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("%") != -1:
        values.iloc[i] = values.iloc[i].replace("%", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float, because we have percentages here and round to .2 after ,

In [ ]:
df["Pct delay due to external causes"] = pd.to_numeric(df["Pct delay due to external causes"].str.replace(",", "."), downcast='float')
df["Pct delay due to external causes"] = df["Pct delay due to external causes"].round(2)

### Pct delay due to infrastructure

Correct the format of the pct delay due to infrastructure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to infrastructure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove every "%"

In [ ]:
values = df["Pct delay due to infrastructure"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("%") != -1:
        values.iloc[i] = values.iloc[i].replace("%", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float because it's percentages and round to 2 after comma

In [ ]:
df["Pct delay due to infrastructure"] = pd.to_numeric(df["Pct delay due to infrastructure"].str.replace(",", "."), downcast='float')
df["Pct delay due to infrastructure"] = df["Pct delay due to infrastructure"].round(2)

### Pct delay due to traffic management

Correct the format of the pct delay due to traffic management

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to traffic management"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove all "%"

In [ ]:
values = df["Pct delay due to traffic management"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("%") != -1:
        values.iloc[i] = values.iloc[i].replace("%", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert all column to float because we have percentages and round to 2 after comma

In [ ]:
df["Pct delay due to traffic management"] = pd.to_numeric(df["Pct delay due to traffic management"].str.replace(",", "."), downcast='float')
df["Pct delay due to traffic management"] = df["Pct delay due to traffic management"].round(2)

### Pct delay due to rolling stock

Correct the format of the pct delay due to rolling stock

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to rolling stock"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove all "%"

In [ ]:
values = df["Pct delay due to rolling stock"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("%") != -1:
        values.iloc[i] = values.iloc[i].replace("%", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float because we have percentages and round of 2 after comma

In [ ]:
df["Pct delay due to rolling stock"] = pd.to_numeric(df["Pct delay due to rolling stock"].str.replace(",", "."), downcast='float')
df["Pct delay due to rolling stock"] = df["Pct delay due to rolling stock"].round(2)

### Pct delay due to station management and equipment reuse

Correct the format of the pct delay due to station management and equipment reuse

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)


In [ ]:
values = df["Pct delay due to station management and equipment reuse"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove "%"

In [ ]:
values = df["Pct delay due to station management and equipment reuse"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("%") != -1:
        values.iloc[i] = values.iloc[i].replace("%", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column because we have percentages and round of 2 after the comma

In [ ]:
df["Pct delay due to station management and equipment reuse"] = pd.to_numeric(df["Pct delay due to station management and equipment reuse"].str.replace(",", "."), downcast='float')
df["Pct delay due to station management and equipment reuse"] = df["Pct delay due to station management and equipment reuse"].round(2)

### Pct delay due to passenger handling (crowding, disabled persons, connections)

Correct the format of the pct delay due to passenger handling (crowding, disabled persons, connections)

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to passenger handling (crowding, disabled persons, connections)"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number and remove the "%"

In [ ]:
values = df["Pct delay due to passenger handling (crowding, disabled persons, connections)"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("%") != -1:
        values.iloc[i] = values.iloc[i].replace("%", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

3. Convert column to float because we have percentages and round of 2 after the comma

In [ ]:
df["Pct delay due to passenger handling (crowding, disabled persons, connections)"] = pd.to_numeric(df["Pct delay due to passenger handling (crowding, disabled persons, connections)"].str.replace(",", "."), downcast='float')
df["Pct delay due to passenger handling (crowding, disabled persons, connections)"] = df["Pct delay due to passenger handling (crowding, disabled persons, connections)"].round(2)

## Dataset after cleaning

In [ ]:
print("Column names and new types are: ")

#We get names of columns in the Dataframe
columns = df.columns
for i in range(len(columns)):
    print(columns[i], end=" -> ")
    print(df[str(columns[i])].dtype)

nb_line, nb_col = df.shape

print(f"\nThere is {nb_col} columns and {nb_line} line in the new dataset.\n- {nb_line_start - nb_line} lines has been removed\n- {nb_col_start - nb_col} columns has been removed.\n {((nb_line_start - nb_line) / nb_line_start * 100):.2f}% of the dataset lines were removed")

display(df)

Exporting the dataset cleaned

In [ ]:
df.to_csv("cleaned_dataset.csv", sep=";", index=False)